# 04 — Análise de experimento (triangulação)
- **Anti-peeking estrutural**: `analyze_experiment` recusa análise antes de `end_date`.
- SCM + ASCM + SDID com p-values de **permutação**; IC **conformal** no SCM
  (ou SDID, via configuração);
  DiD painel com **wild cluster bootstrap** se houver controle fixo.
- Guardrails de razão via **método delta** + correção **Benjamini-Hochberg**.
- Veredito de triangulação: CONCORDANTE / PARCIAL / DIVERGENTE.

In [ ]:
%pip install /Workspace/Shared/supply_experiments/supply_experiments-1.0.3-py3-none-any.whl --quiet --force-reinstall --no-deps
dbutils.library.restartPython()

In [ ]:
from supply_experiments.spark_io import (load_city_panel, TABLES, load_experiment,
                                          NATIONAL_HOLIDAYS)
from supply_experiments.panel import ExperimentWindow
from supply_experiments.reporting import analyze_experiment

dbutils.widgets.text("experiment_id", "")
EXPERIMENT_ID = dbutils.widgets.get("experiment_id")

TARGET_ENV = "sandbox"
tables = TABLES[TARGET_ENV]
rec = load_experiment(spark, tables["registry"], EXPERIMENT_ID)
print(rec.name, "|", rec.status, "|", rec.hypothesis)
print("Regra de decisão pré-registrada:", rec.decision_rule)

In [ ]:
from datetime import timedelta
date_start = str(rec.start_date - timedelta(days=rec.pre_window_days + 7))
date_end   = str(rec.end_date)
panel, _ = load_city_panel(spark, date_start, date_end)

treated = [u["city_norm"] for u in rec.treated_units]
donors  = [u["city_norm"] for u in rec.control_units]
window  = ExperimentWindow(start_date=rec.start_date, end_date=rec.end_date,
                           pre_window_days=rec.pre_window_days)

In [ ]:
# ---- revalidação do controle fixo (habilita o DiD complementar) -------------
# O certificado do notebook 02 só atesta o passado; aqui checamos tendências
# paralelas na janela PRÉ recente deste experimento. Reprovou -> DiD é omitido
# da triangulação (SCM/ASCM/SDID não dependem do controle fixo).
from supply_experiments.spark_io import load_fixed_control
from supply_experiments.design.control_selection import revalidate_fixed_control

did_control = None
fixed = load_fixed_control(spark, tables["registry"])
if fixed:
    val = revalidate_fixed_control(panel, fixed, window_days=90,
                                   holidays=NATIONAL_HOLIDAYS,
                                   target_exclude=set(treated),
                                   end_date=window.pre_end)  # só o pré-período
    print(val.summary())
    if val.passed:
        did_control = [c for c in fixed if c not in set(treated)]
else:
    print("Sem controle fixo no registry — DiD complementar será omitido.")

In [ ]:
report = analyze_experiment(
    panel, rec.experiment_id, treated, donors, window,
    primary_kpi=rec.primary_kpi, guardrail_kpis=rec.guardrail_kpis,
    alpha=0.10, holidays=NATIONAL_HOLIDAYS,
    did_control=did_control,
    run_conformal=True,
)   # levanta ValueError (anti-peeking) se o experimento ainda não terminou
print(report.summary())

In [ ]:
display(report.to_frame())
if not report.guardrails.empty:
    display(report.guardrails)

In [ ]:
# persiste resultados
import json
from datetime import datetime
rows = report.to_frame().assign(
    experiment_id=rec.experiment_id, kpi=rec.primary_kpi,
    triangulation=report.triangulation, analyzed_at=datetime.utcnow(),
    warnings=json.dumps(report.warnings, ensure_ascii=False),
)
spark.createDataFrame(rows).write.format("delta").mode("append") \
    .saveAsTable(tables["results"])
print("Resultados persistidos em", tables["results"])